# 04 — Classificador de adulteração no nível da imagem

Os notebooks anteriores avaliam cada técnica dentro das regiões anotadas. Este notebook responde a outra pergunta: dada uma imagem nova, sem gabarito, ela foi adulterada?

**Regra central.** Nenhuma feature usa as anotações (regiões, caixas, `is_original_area`). O rótulo vem só da coluna `forged` do CSV, e os tipos de adulteração são usados apenas na análise de erros, nunca como entrada do modelo. O classificador é treinado no `train`, o modelo e o limiar são escolhidos no `val`, e o `test` é avaliado uma única vez.

In [1]:
import sys
sys.path.append("../../src")

import warnings

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, f1_score, roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

from dataset_loader import load_dataset
from evaluation import wilson_score_interval, youden_threshold
from features import FEATURE_NAMES, load_or_extract

SEED = 42
DATASET_ROOT = "../../data/findit2"

# NumPy 2.0 com a BLAS Accelerate (macOS) emite avisos espúrios de divisão por zero em
# multiplicações de matrizes. As probabilidades foram conferidas contra um cálculo sem
# BLAS (diferença de 1e-16) e não há NaN nem infinito nas features.
warnings.filterwarnings("ignore", message=".*encountered in matmul", category=RuntimeWarning)


## 1. Dados

Os três splits são carregados com o loader do projeto. O rótulo é `is_tampered`, lido da coluna `forged`. A contagem de imagens CPI usa as anotações e serve apenas para descrever os dados.

In [2]:
amostras = {split: load_dataset(DATASET_ROOT, split) for split in ("train", "val", "test")}

for split, S in amostras.items():
    adulteradas = [s for s in S if s.is_tampered]
    cpi = [s for s in adulteradas if "CPI" in s.tamper_types]
    print(f"{split}: {len(S)} imagens | autênticas: {len(S) - len(adulteradas)} | adulteradas: {len(adulteradas)} | CPI: {len(cpi)}")

aviso: 1 imagens com anotação ausente/malformada tratadas como regions=[] (train)
aviso: 2 imagens com forged divergente da anotação (train): ['X51008114260.png', 'X51006619709.png']
aviso: 2 regiões marcadas como área original mas com modificação real (train)
aviso: 1 imagens listadas em val.txt e ausentes de val/ foram ignoradas: ['X51006619709.png']
aviso: 1 regiões marcadas como área original mas com modificação real (val)
train: 577 imagens | autênticas: 483 | adulteradas: 94 | CPI: 74
val: 192 imagens | autênticas: 159 | adulteradas: 33 | CPI: 26
test: 218 imagens | autênticas: 183 | adulteradas: 35 | CPI: 27


## 2. Features

Sete features por imagem, todas calculadas sobre a imagem inteira, em tons de cinza (`src/features.py`):

| Feature | O que mede | Base no material |
|---|---|---|
| `ela_p95` | percentil 95 do erro da ELA (qualidade 90) | DCT, quantização, JPEG (slides 127, 133, 157, 376-379, 432-440) |
| `copy_move_peak` | maior correlação entre componentes de texto de mesmo tamanho e distantes entre si, como na busca cega do notebook 03 | correlação normalizada e filtro casado (slides 260-272) |
| `ghost_q90_frac` | fração de blocos de texto cujo argmin de qualidade é 90 (0 se não há histórico JPEG) | quantização e DCT |
| `has_jpeg_history` | 1 se a curva de erro da imagem tem vale (histórico JPEG), 0 caso contrário | quantização e DCT |
| `noise_std`, `noise_p95` | desvio padrão e percentil 95 do resíduo (imagem − mediana 3x3) nos pixels de texto | filtro da mediana (slides 201, 208-216) |
| `edge_std` | desvio padrão da magnitude do gradiente de Sobel nos pixels de texto | gradiente e Sobel (slides 225-230) |

Os pixels de texto vêm de uma máscara de Otsu (limiarização: slides 132, 179, 185 e 231; o método de Otsu em si não aparece nos slides). As features são calculadas uma vez e guardadas em `data/features_<split>.csv`.

In [3]:
tabelas = {split: load_or_extract(S, f"../../data/features_{split}.csv") for split, S in amostras.items()}

for split, T in tabelas.items():
    sem = T[T["has_jpeg_history"] == 0]
    print(f"{split}: sem histórico JPEG: {len(sem)} de {len(T)} "
          f"(autênticas: {int((sem['is_tampered'] == 0).sum())}, adulteradas: {int((sem['is_tampered'] == 1).sum())})")

print("\nmédia por classe no train:")
print(tabelas["train"].groupby("is_tampered")[FEATURE_NAMES].mean().rename(index={0: "autêntica", 1: "adulterada"}).T.round(3))

train: sem histórico JPEG: 72 de 577 (autênticas: 56, adulteradas: 16)
val: sem histórico JPEG: 18 de 192 (autênticas: 13, adulteradas: 5)
test: sem histórico JPEG: 30 de 218 (autênticas: 25, adulteradas: 5)

média por classe no train:
is_tampered       autêntica  adulterada
ela_p95               1.609       1.457
copy_move_peak        0.972       0.980
ghost_q90_frac        0.442       0.444
has_jpeg_history      0.884       0.830
noise_std            29.027      28.152
noise_p95            69.030      65.660
edge_std            266.308     260.661


**Interpretação.** As médias por classe no `train` são quase iguais em todas as features: por exemplo, `ela_p95` de 1,61 contra 1,46, `copy_move_peak` de 0,972 contra 0,980, `ghost_q90_frac` de 0,442 contra 0,444 e `edge_std` de 266 contra 261. Isso já indica que nenhuma feature global separa as classes. A adulteração ocupa poucos caracteres, e uma estatística da imagem inteira é dominada pela parte autêntica do documento.

Imagens sem histórico JPEG recebem `ghost_q90_frac = 0` e `has_jpeg_history = 0`: são 72 de 577 no `train` (56 autênticas e 16 adulteradas), 18 de 192 no `val` (13 e 5) e 30 de 218 no `test` (25 e 5).

## 3. Modelos e escolha no `val`

Dois modelos são treinados no `train`, com as features padronizadas (`StandardScaler`, ajustado só no `train`):

- regressão logística com `class_weight="balanced"`, que compensa o desbalanceamento das classes;
- árvore de decisão com profundidade máxima 3.

No `val`, cada modelo produz uma probabilidade de adulteração por imagem. Para cada modelo, o limiar de decisão é o que maximiza o índice de Youden (J = sensibilidade + especificidade − 1). Fica o modelo com maior J no `val`. O F1 é reportado para comparação.

In [4]:
X = {split: T[FEATURE_NAMES].to_numpy() for split, T in tabelas.items()}
y = {split: T["is_tampered"].to_numpy() for split, T in tabelas.items()}

modelos = {
    "logística": make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=1000, random_state=SEED)),
    "árvore (prof. 3)": make_pipeline(StandardScaler(), DecisionTreeClassifier(max_depth=3, random_state=SEED)),
}

escolha = {}
for nome, modelo in modelos.items():
    modelo.fit(X["train"], y["train"])
    prob = modelo.predict_proba(X["val"])[:, 1]
    limiar, tpr, fpr = youden_threshold(prob[y["val"] == 1], prob[y["val"] == 0], np.unique(prob))
    f1 = f1_score(y["val"], prob >= limiar)
    escolha[nome] = {"limiar": limiar, "J": tpr - fpr, "tpr": tpr, "fpr": fpr, "f1": f1,
                     "auc": roc_auc_score(y["val"], prob)}
    print(f"{nome}: AUC val = {escolha[nome]['auc']:.3f} | limiar = {limiar:.3f} | sensibilidade = {tpr:.1%} | "
          f"falso positivo = {fpr:.1%} | J = {tpr - fpr:+.3f} | F1 = {f1:.3f}")

MODELO = max(escolha, key=lambda m: escolha[m]["J"])
LIMIAR = escolha[MODELO]["limiar"]
print(f"\nescolhido no val: {MODELO}, limiar = {LIMIAR:.3f}")

logística: AUC val = 0.569 | limiar = 0.443 | sensibilidade = 93.9% | falso positivo = 74.2% | J = +0.197 | F1 = 0.341
árvore (prof. 3): AUC val = 0.516 | limiar = 0.333 | sensibilidade = 12.1% | falso positivo = 3.1% | J = +0.090 | F1 = 0.190

escolhido no val: logística, limiar = 0.443


**Interpretação.** No `val`, a AUC é 0,569 para a logística e 0,516 para a árvore, ambas próximas de 0,5, que é o valor de uma ordenação ao acaso. A logística tem o maior J (+0,197) e foi escolhida. Seu limiar de 0,443 dá sensibilidade de 93,9%, mas falso positivo de 74,2%: o critério de Youden escolhe marcar quase tudo, porque nenhum limiar separa bem as classes.

## 4. Avaliação no `test`

O modelo e o limiar escolhidos no `val` são aplicados uma única vez ao `test`. A comparação de referência é o classificador trivial que diz "autêntica" para toda imagem.

In [5]:
modelo = modelos[MODELO]
prob_test = modelo.predict_proba(X["test"])[:, 1]
pred = (prob_test >= LIMIAR).astype(int)
vn, fp, fn, vp = confusion_matrix(y["test"], pred, labels=[0, 1]).ravel()

sens, espec = vp / (vp + fn), vn / (vn + fp)
precisao = vp / (vp + fp) if vp + fp else float("nan")
s0, s1 = wilson_score_interval(sens, vp + fn)
e0, e1 = wilson_score_interval(espec, vn + fp)

print(f"modelo: {MODELO} | limiar: {LIMIAR:.3f}")
print("matriz de confusão (linhas = verdade, colunas = previsão):")
print(pd.DataFrame([[vn, fp], [fn, vp]], index=["autêntica", "adulterada"], columns=["prev. autêntica", "prev. adulterada"]))
print(f"\nprecisão:        {precisao:.1%}")
print(f"sensibilidade:   {sens:.1%}  (IC 95%: {s0:.1%} a {s1:.1%})")
print(f"especificidade:  {espec:.1%}  (IC 95%: {e0:.1%} a {e1:.1%})")
print(f"acurácia:        {(vp + vn) / len(pred):.1%}")
print(f"AUC:             {roc_auc_score(y['test'], prob_test):.3f}")
print(f"\nbaseline (tudo autêntica): acurácia = {(y['test'] == 0).mean():.1%}, sensibilidade = 0%")

modelo: logística | limiar: 0.443
matriz de confusão (linhas = verdade, colunas = previsão):
            prev. autêntica  prev. adulterada
autêntica                43               140
adulterada                6                29

precisão:        17.2%
sensibilidade:   82.9%  (IC 95%: 67.3% a 91.9%)
especificidade:  23.5%  (IC 95%: 17.9% a 30.1%)
acurácia:        33.0%
AUC:             0.545

baseline (tudo autêntica): acurácia = 83.9%, sensibilidade = 0%


**Interpretação.** No `test`, o classificador marca 169 de 218 imagens como adulteradas. Detecta 29 das 35 adulteradas (sensibilidade de 82,9%, IC 95%: 67,3% a 91,9%), mas acerta só 43 das 183 autênticas (especificidade de 23,5%, IC 95%: 17,9% a 30,1%). A precisão é de 17,2%, praticamente igual à proporção de adulteradas no `test` (35 de 218, 16,1%). A AUC é 0,545. A acurácia (33,0%) fica muito abaixo do baseline que chama tudo de autêntico (83,9%). Em resumo, o desempenho é próximo do acaso.

### Peso de cada feature

Para a regressão logística, os coeficientes são comparáveis entre si porque as features estão padronizadas: coeficiente positivo empurra para "adulterada". Para a árvore, a importância mede quanto cada feature reduz a impureza nas divisões.

In [6]:
logistica = modelos["logística"][-1]
arvore = modelos["árvore (prof. 3)"][-1]
pesos = pd.DataFrame({"coef. logística": logistica.coef_[0], "importância árvore": arvore.feature_importances_},
                     index=FEATURE_NAMES).round(3)
print(pesos.sort_values("coef. logística", key=np.abs, ascending=False))

                  coef. logística  importância árvore
copy_move_peak              0.307               0.704
has_jpeg_history           -0.176               0.000
ela_p95                    -0.155               0.036
edge_std                   -0.119               0.000
noise_p95                   0.042               0.035
ghost_q90_frac              0.038               0.224
noise_std                  -0.005               0.000


**Interpretação.** O `copy_move_peak` é a feature de maior peso nos dois modelos (coeficiente de +0,307 na logística e importância de 0,704 na árvore). Mesmo assim, ele pouco separa as classes, porque caracteres repetidos já levam o pico para perto de 1 nas autênticas (média de 0,972 no `train`).

### Erros por tipo de adulteração

A sensibilidade é separada pelo tipo de adulteração anotado (só para análise). Uma imagem com mais de um tipo entra em cada um deles.

In [7]:
T = tabelas["test"].assign(pred=pred)
adult = T[T["is_tampered"] == 1]
for tipo in ("CPI", "CPO", "IMI", "CUT", "PIX"):
    grupo = adult[adult["tamper_types"].fillna("").str.split("+").apply(lambda ts: tipo in ts)]
    if len(grupo):
        print(f"{tipo}: detectadas {int(grupo['pred'].sum())} de {len(grupo)}")
print(f"\nautênticas marcadas como adulteradas: {int(((T['is_tampered'] == 0) & (T['pred'] == 1)).sum())} de {int((T['is_tampered'] == 0).sum())}")

CPI: detectadas 22 de 27
CPO: detectadas 2 de 2
IMI: detectadas 3 de 3
CUT: detectadas 3 de 5
PIX: detectadas 1 de 3

autênticas marcadas como adulteradas: 140 de 183


## 5. Conclusão

**Resultado.** Com sete features globais, sem uso das anotações, o melhor modelo escolhido no `val` (regressão logística, limiar 0,443) tem desempenho próximo do acaso no `test`: AUC de 0,545, precisão de 17,2% (contra 16,1% de adulteradas) e especificidade de 23,5%. Ele detecta a maioria das adulteradas só porque marca 77% das imagens como adulteradas. Na prática, não é utilizável como classificador.

**Por que falha.** As adulterações do dataset ocupam poucos caracteres, e cada feature resume a imagem inteira. Nesse resumo, a pequena região alterada fica diluída no restante autêntico, e as médias por classe ficam quase iguais. A feature mais promissora, o pico de copy-move, sofre com o texto repetido legítimo (dígitos e letras da mesma fonte), que gera correlação alta também em documentos autênticos, como mostrou a busca cega do notebook 03. A ELA global já não separava as classes no notebook 01, e o mesmo se repete aqui.

**O que escapa.** Os tipos que o modelo mais deixa passar no `test` são PIX (1 de 3 detectados) e CUT (3 de 5). A modificação à mão com pincel e a remoção de caracteres não criam cópias nem mudam o histórico de compressão de forma visível na imagem inteira. Esses números vêm de grupos muito pequenos e devem ser lidos com cautela. As taxas de detecção de CPI (22 de 27), CPO (2 de 2) e IMI (3 de 3) não indicam bom desempenho, porque o limiar marca quase todas as imagens, inclusive 140 das 183 autênticas.

**Leitura para o relatório.** As técnicas dos notebooks 01 a 03 mostram sinal quando a região suspeita é conhecida, mas resumidas em estatísticas globais da imagem não distinguem documentos adulterados de autênticos neste dataset. Esse resultado negativo é parte da conclusão do trabalho. Uma melhoria possível, não feita, seria usar features locais (por exemplo, o valor máximo por bloco em vez da estatística da imagem inteira), o que exige uma etapa de localização e uma avaliação própria.
